<a href="https://colab.research.google.com/github/adilsaleem007/Python-Basics-Assignment/blob/main/intermediate_assessment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

from xgboost import XGBClassifier

In [17]:
train = pd.read_csv("/content/train_LZdllcl.csv")
test = pd.read_csv("/content/test_2umaH9m.csv")
sample = pd.read_csv("/content/sample_submission_M0L0uXE.csv")



In [18]:
train.head()
train.shape
train.isnull().sum()


train['is_promoted'].value_counts()

,count
is_promoted,
0,50140
1,4668


In [19]:
train['education'].fillna("Bachelor's", inplace=True)
train['previous_year_rating'].fillna(0, inplace=True)

/tmp/ipykernel_1186/1565194102.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train['education'].fillna("Bachelor's", inplace=True)
/tmp/ipykernel_1186/1565194102.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)',

In [20]:
test['education'].fillna("Bachelor's", inplace=True)
test['previous_year_rating'].fillna(0, inplace=True)

/tmp/ipykernel_1186/3785493461.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  test['education'].fillna("Bachelor's", inplace=True)
/tmp/ipykernel_1186/3785493461.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', 

In [21]:
train['is_senior'] = (train['length_of_service'] > 5).astype(int)
test['is_senior'] = (test['length_of_service'] > 5).astype(int)

train['high_score'] = (train['avg_training_score'] > 70).astype(int)
test['high_score'] = (test['avg_training_score'] > 70).astype(int)

train['training_per_age'] = train['avg_training_score'] / train['age']
test['training_per_age'] = test['avg_training_score'] / test['age']

In [22]:
train = pd.get_dummies(train)
test = pd.get_dummies(test)


train, test = train.align(test, join='left', axis=1, fill_value=0)

In [23]:
X = train.drop(['is_promoted', 'employee_id'], axis=1)
y = train['is_promoted']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [24]:
model = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=9,
    random_state=42
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=400,
              n_jobs=None, num_parallel_tree=None, ...)

In [25]:
y_prob = model.predict_proba(X_val)[:,1]

best_score = 0
best_thresh = 0

for t in [0.2, 0.25, 0.3, 0.35, 0.4]:
    y_pred = (y_prob > t).astype(int)
    score = f1_score(y_val, y_pred)

    print(t, score)

    if score > best_score:
        best_score = score
        best_thresh = t

print("Best Threshold:", best_thresh)

0.2 0.332319391634981
0.25 0.3443548387096774
0.3 0.35368956743002544
0.35 0.36703445170006754
0.4 0.38213518032003824
Best Threshold: 0.4


In [26]:
model.fit(X, y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=400,
              n_jobs=None, num_parallel_tree=None, ...)

In [31]:

test_final = test[X.columns]

test_prob = model.predict_proba(test_final)[:,1]
test_pred = (test_prob > best_thresh).astype(int)

In [32]:
print(X.shape)
print(test.shape)

(54808, 61)
(23490, 63)


In [33]:
sample['is_promoted'] = test_pred
sample.to_csv("final_submission.csv", index=False)

In [34]:
from google.colab import files
files.download("final_submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>